# JP Morgan Chase — AI Finance Agent
## Lab 04 : Claude AI Integration — Conversational Agent
**Author :** Fabrice William FOMHOM  
**Date :** March 2026  
**Objective :** Connect our financial analysis to Claude AI 
to build a conversational agent that answers business questions

## What We Build
1. Connect securely to Claude API
2. Build a Finance Agent with financial context
3. Ask business questions in natural language
4. Integrate our ML fraud model with Claude

In [12]:
# ============================================================
# Step 2 : First test — Talk to Claude
# ============================================================

def ask_claude(question, context=""):
    message = client.messages.create(
        model      = "claude-sonnet-4-20250514",
        max_tokens = 1024,
        system     = "You are a senior Financial Analyst at JP Morgan Chase. You analyze financial data, detect fraud patterns, and provide executive-level insights. Be concise, professional, and data-driven.",
        messages   = [
            {
                "role"   : "user",
                "content": f"{context}\n\n{question}" if context else question
            }
        ]
    )
    return message.content[0].text

# First test
print("=" * 60)
print("JP MORGAN — AI FINANCE AGENT")
print("=" * 60)

response = ask_claude(
    "What are the top 3 red flags that indicate credit card fraud?"
)

print(f"\n🤖 Agent : {response}")

JP MORGAN — AI FINANCE AGENT


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CZYqyGeAGHV2XoLMXtEHS'}

In [3]:
# ============================================================
# JP Morgan Chase — Lab 04 : Claude AI Agent
# Step 1 : Import libraries and connect to Claude API
# ============================================================

import os
import anthropic
import pandas as pd
import numpy as np

# Set API key directly (safe on your local machine)
os.environ["ANTHROPIC_API_KEY"] = "YOUR_API_KEY_HERE"

# Initialize Claude client
client = anthropic.Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

print("✅ Libraries imported successfully")
print("✅ API key set successfully")
print("✅ Claude client initialized")

✅ Libraries imported successfully
✅ API key set successfully
✅ Claude client initialized


In [17]:
# Verify API key is set correctly
key = os.environ.get("ANTHROPIC_API_KEY", "NOT FOUND")
print(f"Key starts with : {key[:20]}...")
print(f"Key length      : {len(key)} characters")

Key starts with : YOUR_API_KEY_HERE
Key length      : 108 characters


In [19]:
# Simple connection test
try:
    test = client.messages.create(
        model      = "claude-haiku-4-5-20251001",
        max_tokens = 100,
        messages   = [
            {"role": "user", "content": "Say: JP Morgan Agent connected!"}
        ]
    )
    print("✅ Claude API connected successfully!")
    print(f"🤖 Response : {test.content[0].text}")
except Exception as e:
    print(f"❌ Error : {e}")

✅ Claude API connected successfully!
🤖 Response : JP Morgan Agent connected!


In [20]:
# ============================================================
# Step 3 : Build the JP Morgan Finance Agent
# Give Claude context about our financial data
# ============================================================

# Load our dataset
df = pd.read_csv(
    r"C:\Users\HP\jpmorgan_finance_agent\data\jpmorgan_transactions.csv",
    parse_dates=["date"]
)

# Build financial context summary
context = f"""
You are analyzing JP Morgan Chase transaction data with these facts:

PORTFOLIO SUMMARY:
- Total transactions  : {len(df):,}
- Total volume        : ${df['amount'].sum():,.2f}
- Average transaction : ${df['amount'].mean():,.2f}
- Date range          : {df['date'].min().date()} to {df['date'].max().date()}

FRAUD SUMMARY:
- Total fraud cases   : {df['is_fraud'].sum()}
- Fraud rate          : {df['is_fraud'].mean()*100:.2f}%
- Fraud amount        : ${df[df['is_fraud']==1]['amount'].sum():,.2f}
- Avg fraud amount    : ${df[df['is_fraud']==1]['amount'].mean():,.2f}
- Avg normal amount   : ${df[df['is_fraud']==0]['amount'].mean():,.2f}

TOP FRAUD CATEGORIES:
{df.groupby('category')['is_fraud'].mean().sort_values(ascending=False).head(3).to_string()}

MONTHLY FRAUD (top 3 months):
{df.groupby('month_name')['is_fraud'].sum().sort_values(ascending=False).head(3).to_string()}
"""

print("✅ Financial context built successfully")
print(f"   Context length : {len(context)} characters")
print("\nContext preview :")
print(context)

✅ Financial context built successfully
   Context length : 594 characters

Context preview :

You are analyzing JP Morgan Chase transaction data with these facts:

PORTFOLIO SUMMARY:
- Total transactions  : 1,000
- Total volume        : $165,365.53
- Average transaction : $165.37
- Date range          : 2024-01-03 to 2024-12-31

FRAUD SUMMARY:
- Total fraud cases   : 29
- Fraud rate          : 2.90%
- Fraud amount        : $23,736.73
- Avg fraud amount    : $818.51
- Avg normal amount   : $145.86

TOP FRAUD CATEGORIES:
category
restaurant         0.075758
travel             0.031496
online_shopping    0.030534

MONTHLY FRAUD (top 3 months):
month_name
Apr    4
Jul    4
Jun    4



In [21]:
# ============================================================
# Step 4 : Ask the Finance Agent real business questions
# ============================================================

def ask_finance_agent(question):
    """Ask our JP Morgan Finance Agent a question"""
    print(f"❓ Question : {question}")
    print("-" * 60)
    response = ask_claude(question, context)
    print(f"🤖 Agent    : {response}")
    print("=" * 60)
    print()

# ── Ask 4 real business questions ───────────────────────────
ask_finance_agent(
    "What is the overall fraud risk level of this portfolio and what are the top 2 recommendations?"
)

ask_finance_agent(
    "Which merchant category should JP Morgan flag for immediate review and why?"
)

ask_finance_agent(
    "A transaction of $950 just came in from a restaurant. Should we flag it?"
)

ask_finance_agent(
    "Write a 3-sentence executive summary of this portfolio for the CEO."
)

❓ Question : What is the overall fraud risk level of this portfolio and what are the top 2 recommendations?
------------------------------------------------------------
🤖 Agent    : ## FRAUD RISK ASSESSMENT: **MODERATE-HIGH**

**Key Risk Indicators:**
- Fraud rate of 2.90% is **significantly above industry benchmark** (~0.5-1.0%)
- Average fraud loss of $818.51 is **5.6x higher** than normal transactions ($145.86)
- Total fraud exposure: $23,736.73 (14.4% of portfolio volume)

## TOP 2 RECOMMENDATIONS:

### 1. **IMPLEMENT ENHANCED RESTAURANT TRANSACTION MONITORING**
- **Priority:** Immediate
- **Rationale:** Restaurant category shows highest fraud concentration at 7.6% (2.5x portfolio average)
- **Action:** Deploy real-time alerts for restaurant transactions >$300 and multiple restaurant charges within 2-hour windows

### 2. **STRENGTHEN Q2 FRAUD PREVENTION CONTROLS**
- **Priority:** High  
- **Rationale:** April-June shows consistent elevated fraud activity (4 cases/month vs. 2.4 aver